In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 8 - WEEK 10 BAYESIAN OPTIMISATION
# Run from inside the week10/ folder
# ============================================================
#
# Strategy:
# - Refit ARD Matern GP including Week 9.
# - Check Week 9 realised calibration.
# - Search local + moderate-wide + global pools.
# - Compare EI, posterior mean and UCB.
# - Prefer the established high-value basin unless global
#   exploration is supported by both mean and uncertainty.
# ============================================================


# ------------------------------------------------------------
# 1. Load Week 10 cumulative data
# ------------------------------------------------------------

X = np.load("function8/initial_inputs.npy")
Y = np.load("function8/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. WEEK 9 CALIBRATION CHECK
# ------------------------------------------------------------
#
# Week 9 selected:
# [0.11706245, 0.18313520, 0.10430401, 0.12041429,
#  0.77064421, 0.49765153, 0.17958835, 0.55885207]
#
# Predicted:
# mean ≈ 9.996569
# std  ≈ 0.037622
#
# Actual:
# 9.9940246476926
# ------------------------------------------------------------

week9_pred_mean = 9.996569
week9_pred_std = 0.037622
week9_actual = 9.9940246476926

week9_error = (
    week9_actual
    - week9_pred_mean
)

week9_z_error = (
    week9_error
    / week9_pred_std
)

print("\n================================")
print("WEEK 9 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week9_pred_mean)
print("Predicted std :", week9_pred_std)
print("Actual        :", week9_actual)

print("\nPrediction error:")
print(week9_error)

print("\nError / predicted std:")
print(week9_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(8) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Candidate generation
# ------------------------------------------------------------
#
# Week 9 was extremely well calibrated and produced a new best,
# so concentrate heavily around the new incumbent.
#
# Still retain wider + global pools for diagnostics.
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.20 * lengthscales,
    0.015,
    0.08
)

wide_scale = np.clip(
    0.40 * lengthscales,
    0.04,
    0.15
)

print("\n================================")
print("CANDIDATE SCALES")
print("================================")

print("Local widths:")
print(local_scale)

print("\nWide widths:")
print(wide_scale)


local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(180000, 8)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(140000, 8)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(130000, 8)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])


# ------------------------------------------------------------
# 6. Remove near-duplicates
# ------------------------------------------------------------

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 7. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 8. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 9. EI sensitivity
# ------------------------------------------------------------

y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi=xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 10. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 11. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 12. Distance from incumbent
# ------------------------------------------------------------

def distance_from_best(x):
    return np.linalg.norm(
        x - best_x
    )

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    distance_from_best(
        candidates[ei_idx]
    )
)

print(
    "Highest mean:",
    distance_from_best(
        candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        distance_from_best(
            candidates[idx]
        )
    )


# ------------------------------------------------------------
# 13. Domain-boundary diagnostic
# ------------------------------------------------------------

def domain_boundary_status(
    x,
    tol=0.01
):

    status = []

    for j in range(len(x)):

        if x[j] <= tol:
            status.append(
                f"x{j+1}~0"
            )

        elif x[j] >= 1.0 - tol:
            status.append(
                f"x{j+1}~1"
            )

    if not status:
        return "interior"

    return ", ".join(status)


print("\n================================")
print("DOMAIN BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    domain_boundary_status(
        candidates[ei_idx]
    )
)

print(
    "Highest mean:",
    domain_boundary_status(
        candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        domain_boundary_status(
            candidates[idx]
        )
    )

DATA
X shape: (49, 8)
Y shape: (49,)

Current best:
[0.117062 0.183135 0.104304 0.120414 0.770644 0.497652 0.179588 0.558852] -> 9.9940246476926

Y range:
min = 5.5921933895401965
max = 9.9940246476926
std = 1.1878748602460514

WEEK 9 CALIBRATION CHECK
Predicted mean: 9.996569
Predicted std : 0.037622
Actual        : 9.9940246476926

Prediction error:
-0.0025443523073995777

Error / predicted std:
-0.06762937396734829


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/co


GP FIT

Fitted kernel:
0.882**2 * Matern(length_scale=[1.26, 2, 0.996, 2, 2, 2, 1.36, 2], nu=2.5) + WhiteKernel(noise_level=1e-08)

ARD lengthscales:
[1.25937213 2.         0.99560409 2.         2.         2.
 1.36101313 2.        ]

Normalised inverse-lengthscale sensitivity:
[0.15776149 0.09934021 0.19955766 0.09934021 0.09934021 0.09934021
 0.1459798  0.09934021]

CANDIDATE SCALES
Local widths:
[0.08 0.08 0.08 0.08 0.08 0.08 0.08 0.08]

Wide widths:
[0.15 0.15 0.15 0.15 0.15 0.15 0.15 0.15]

Candidates after duplicate filtering:
450000

PRIMARY EI
candidate = [0.         0.         0.         0.09510388 1.         0.18517193
 0.19599051 0.29529268]
mean = 9.810128024197676
std = 0.197349169352096
EI = 0.01869002038413157

EI SENSITIVITY

xi = 0.000000e+00 
 candidate = [0.         0.         0.         0.09510388 1.         0.18517193
 0.19599051 0.29529268] 
 mean = 9.810128 
 std = 0.197349 
 EI = 0.01869002 

xi = 1.187875e-02 
 candidate = [0.         0.         0.         0.09

In [2]:
# ============================================================
# FINAL FUNCTION 8 - WEEK 10 SELECTION
# ============================================================
#
# Week 9 calibration was excellent (~ -0.068 sigma) and
# produced a new observed best.
#
# Global EI is rejected because it is uncertainty-driven,
# far from the incumbent, and pushed onto several boundaries.
#
# Highest posterior mean and UCB beta = 0.05 and 0.1 all
# select the same local candidate.
#
# Since recent local calibration is very strong, prefer the
# exploitative consensus point over higher-beta exploration.

beta = 0.1

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week10_candidate = candidates[final_idx]

print("Week 10 Function 8 candidate:")
print(week10_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week10_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week10_candidate
)

print("\nPortal format:")
print(portal)

Week 10 Function 8 candidate:
[0.14618732 0.18717285 0.14395454 0.09824329 0.77845214 0.49762671
 0.17075698 0.59484327]

Predicted mean:
9.997490834716594

Predicted std:
0.018147866338162684

UCB:
9.999305621350409

Distance from current best:
0.06605013543774699

Portal format:
0.146187-0.187173-0.143955-0.098243-0.778452-0.497627-0.170757-0.594843
